<a href="https://colab.research.google.com/github/MarthaBBrito/DataScience/blob/main/RecomendadorPresentes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Recomendador de Presentes

Esse projeto nasceu da idéia de facilitar a criação da lista de presentes de um casamento ou chá de panela. Como esse é um dos meus objetivos, decidi brincar com as possibilidades enquanto esse momento não chega.

#Criando um modelo conectado à uma API

Nesse primeiro momento será testado como o modelo responde quando utiliza dados de uma API para gerar uma lista.

##Configurando o Dataset

In [15]:
# Bibliotecas essenciais
import requests
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

#Dados da API
url = "https://fakestoreapi.com/products"
response = requests.get(url)
data = response.json()
#Transformar em DataFrame
df = pd.DataFrame(data)
df = df[['title', 'category', 'price', 'description']]
df.head()

,title,category,price,description
0,"Fjallraven - Foldsack No. 1 Backpack, Fits 15 ...",men's clothing,109.95,Your perfect pack for everyday use and walks i...
1,Mens Casual Premium Slim Fit T-Shirts,men's clothing,22.30,"Slim-fitting style, contrast raglan long sleev..."
2,Mens Cotton Jacket,men's clothing,55.99,great outerwear jackets for Spring/Autumn/Wint...
3,Mens Casual Slim Fit,men's clothing,15.99,The color could be slightly different between ...
4,John Hardy Women's Legends Naga Gold & Silver ...,jewelery,695.00,"From our Legends Collection, the Naga was insp..."


##Criando o modelo de Recomendação

In [16]:
class APIGiftRecommender:
  def __init__(self, df):
    self.data = df.copy()
    self.vectorizer = TfidfVectorizer(stop_words='english')
    self._prepare()

  def _prepare(self):
    self.data['text'] = (
        self.data['title'] + ' ' +
        self.data['category'] + ' ' +
        self.data['description']
    )
    self.matrix = self.vectorizer.fit_transform(self.data['text'])

  def recommend(self, interest, price_max=None, n=5):
    query_vec = self.vectorizer.transform([interest])
    sim_scores = cosine_similarity(query_vec, self.matrix).flatten()
    self.data['similarity'] = sim_scores

    recs = self.data.sort_values('similarity', ascending=False)
    if price_max:
      recs = recs[recs['price'] <= price_max]

    return recs[['title', 'category', 'price','description' ]].head(n)

##Testando o Modelo

In [20]:
recommender = APIGiftRecommender(df)
recomendacoes = recommender.recommend("fancy wardrobe", price_max=50)
recomendacoes

,title,category,price,description
1,Mens Casual Premium Slim Fit T-Shirts,men's clothing,22.30,"Slim-fitting style, contrast raglan long sleev..."
3,Mens Casual Slim Fit,men's clothing,15.99,The color could be slightly different between ...
6,White Gold Plated Princess,jewelery,9.99,Classic Created Wedding Engagement Solitaire D...
7,Pierced Owl Rose Gold Plated Stainless Steel D...,jewelery,10.99,Rose Gold Plated Double Flared Tunnel Plug Ear...
15,Lock and Love Women's Removable Hooded Faux Le...,women's clothing,29.95,100% POLYURETHANE(shell) 100% POLYESTER(lining...


##Considerações:

O modelo é promissor, mas fortemente ligado ao dataset. Como a API integrada é limitada, a lista também é limitada. Uma opção viável seria criar um dataset próprio.

#Criando um modelo com Dataset próprio

Dada as considerações anteriores, serão feitos testes com um dataset próprio, conectado a um arquivo `.csv`